# Signal

> Signal Processing Helper Functions

In [ ]:
#| default_exp signal

In [ ]:
#| hide
from nbdev.showdoc import *
import glob, zarr

In [ ]:
#| export
from scipy import signal, interpolate
import torch, numpy as np, torch.nn.functional as F
from fractions import Fraction

In [ ]:
#| export
def butterworth(waveform_array, # waveform array of shape (n_samples,)
                freq_range, # e.g. [0.5, 8] for bandpass, 0.5 for highpass, 8 for lowpass
                btype, # 'bandpass', 'lowpass', 'highpass', 'bandstop'
                fs=128, # sampling frequency
                order=4 # filter order (default 4)
                ): 
    """
    Butterworth filter

    Returns: filtered waveform array of shape (n_samples,)
    """
    sos = signal.butter(order, freq_range, fs=fs, btype=btype, output='sos')
    filtered = signal.sosfiltfilt(sos, waveform_array) # zero phase filter (no phase shift)
    return filtered

def resample_waveform(waveform_array, # waveform array of shape (n_samples,)
                      fs_in, # original sampling frequency
                      fs_out, # desired sampling frequency
                      is_spo2=False # if True, use linear interpolation (for SpO2 signal or other low-sampling-rate signals)
                      ):
    """
    Resample waveform to desired sampling frequency
    
    Returns: resampled waveform array of shape (n_resampled_samples,)
    """
    if not is_spo2:
        resample_fraction = Fraction(fs_out, fs_in)#.limit_denominator(100)
        resampled_waveform = signal.resample_poly(waveform_array, resample_fraction.numerator, resample_fraction.denominator)
    else:
        # linear interpolation
        t = np.arange(0, len(waveform_array)*(1/fs_in), 1/fs_in)
        resample_f = interpolate.make_interp_spline(t, waveform_array, k=1) # linear interpolation
        t_new = np.arange(0, len(waveform_array)*(1/fs_in), 1/fs_out)
        resampled_waveform = resample_f(t_new)
    return resampled_waveform


def iir_filter(waveform_array, # waveform array of shape (n_samples,),
               freq_range, # e.g. [0.5, 8] for bandpass, 0.5 for highpass, 8 for lowpass
               btype,  # 'bandpass', 'lowpass', 'highpass', 'bandstop'
               order=16,  # filter order (default 16)
               fs=128  # sampling frequency (default 128)
               ):
    """
    IIR filter using elliptic filter design
    as described in https://www.researchsquare.com/article/rs-6307069/v1

    Returns: filtered waveform array of shape (n_samples,)
    """
    sos = signal.iirfilter(N=order, Wn=freq_range, rp=1, rs=40, btype=btype, analog=False, ftype='ellip', output='sos', fs=fs)
    filtered_data = signal.sosfiltfilt(sos, waveform_array)
    return filtered_data

def iqr_normalization(waveform_array, # waveform array of shape (n_samples,)
                      is_spo2=False # if True, use SpO2 normalization (0.6-1.0 scaled to -1 to 1)
                      ):
    """
    IQR normalization to scale waveform to -1 to 1
    For SpO2, scale 0.6-1.0 to -1 to 1
    
    Returns: normalized waveform array of shape (n_samples,)
    """
    eps = 1e-10
    if not is_spo2:
        q5 = np.percentile(waveform_array, 5)
        q95 = np.percentile(waveform_array, 95)
        waveform_array = 2*(waveform_array - q5) / (q95 - q5 + eps) - 1 # scaled to -1 to 1
    else:
        # spo2 is scaled to 0.60-1.0
        if np.mean(waveform_array) > 1.0:
            waveform_array = waveform_array / 100.
        waveform_array = waveform_array.clip(0.6, 1.0)
        waveform_array = 2*(waveform_array - 0.6) / (1.0 - 0.6) - 1 # scaled to -1 to 1
    return waveform_array

In [ ]:
#| export
def wabp_onset_detector(abp_signal, fs=125):
    """
    ABP onset detector adapted from physionet at: https://physionet.org/content/cardiac-output/1.0.0/code/2analyze/wabp.m
    Detects onset of each beat in ABP waveform.

    This was written for a 125 Hz ABP signal. Specific params could likely be adjusted for different frequencies.
    
    Args:
        abp_signal: ABP waveform in mmHg
        fs: sampling frequency (default 125 Hz)
    
    Returns:
        onset_indices: array of onset sample indices
    """
    if len(abp_signal) < 1000:
        return np.array([])
    
    # Scale physiologic ABP (adapted from original)
    offset = 1600
    scale = 20
    Araw = abp_signal * scale - offset
    
    # Low-pass filter (approximating the original filter)
    # Original: [1 0 0 0 0 -2 0 0 0 0 1] / [1 -2 1] / 24 + 30
    b = np.array([1, 0, 0, 0, 0, -2, 0, 0, 0, 0, 1])
    a = np.array([1, -2, 1])
    A_filt = signal.lfilter(b, a, Araw) / 24 + 30
    
    # Compensate for group delay and rescale
    A = (A_filt[4:] + offset) / scale
    
    # Slope-sum function
    dypos = np.diff(A)
    dypos[dypos < 0] = 0
    
    # Convolve with 16-sample window (original uses conv(ones(16,1), dypos))
    ssf = np.convolve(np.ones(16), dypos, mode='full')
    ssf = np.concatenate([[0, 0], ssf])  # Pad to match original indexing % Produce analysing window of 128 ms
    
    # Decision rule initialization
    if len(ssf) < (fs*10):
        # less than 10 secs of data so don't perform beat detection
        return np.array([])
    
    avg0 = np.mean(ssf[:int(8*fs)])  # Average of first 8 seconds
    threshold0 = 3 * avg0       # Initial decision threshold
    
    # Detection loop
    lockout = 0
    timer = 0
    onsets = []
    
    for t in range(50, len(ssf) - 17):
        lockout = max(0, lockout - 1)
        timer += 1
        
        # Check for detection
        if lockout == 0 and ssf[t] > avg0 + 5:
            timer = 0
            
            # Find local max and min of SSF
            maxSSF = np.max(ssf[t:t+17])
            minSSF = np.min(ssf[t-16:t+1])
            
            if maxSSF > (minSSF + 10):
                onset_threshold = 0.01 * maxSSF
                
                # Find onset time
                tt = np.arange(t-16, t+1)
                dssf = ssf[tt] - ssf[tt-1]
                
                # Find where derivative last falls below onset threshold
                below_onset = np.where(dssf < onset_threshold)[0]
                if len(below_onset) > 0:
                    beat_time = below_onset[-1] + t - 17
                    onsets.append(beat_time)
                    
                    # Adjust threshold (learning)
                    threshold0 = threshold0 + 0.1 * (maxSSF - threshold0)
                    avg0 = threshold0 / 3
                    
                    lockout = 32  # Refractory period (~250ms at 125Hz)
        
        # Lower threshold if no detection for too long
        if timer > 312:  # ~2.5 seconds at 125Hz
            threshold0 = max(threshold0 - 1, avg0)
            avg0 = threshold0 / 3
    
    return np.array(onsets) - 2  # Adjust for original offset  


def abp_features(abp, onset_times, fs=125):
    """
    ABP waveform feature extractor. From physionet: https://physionet.org/content/cardiac-output/1.0.0/code/2analyze/abpfeature.m
    
    Args:
        abp: ABP waveform (sampled at fs Hz)
        onset_times: Array of onset times in samples
        fs: Sampling frequency (default 125 Hz)
    
    Returns:
        features: Array with beat-to-beat ABP features (12 columns)
            Col 0:  Time of systole   [samples]
            Col 1:  Systolic BP       [mmHg]
            Col 2:  Time of diastole  [samples]
            Col 3:  Diastolic BP      [mmHg]
            Col 4:  Pulse pressure    [mmHg]
            Col 5:  Mean pressure     [mmHg]
            Col 6:  Beat Period       [samples]
            Col 7:  mean_dyneg
            Col 8:  End of systole time  0.3*sqrt(RR)  method
            Col 9:  Area under systole   0.3*sqrt(RR)  method
            Col 10: End of systole time  1st min-slope method
            Col 11: Area under systole   1st min-slope method
            Col 12: HR
    """
    
    if len(onset_times) < 2:
        return np.array([])
    
    # P_sys, P_dias
    window = 40
    OT = onset_times[:-1]  # All onsets except last
    beat_qty = len(OT)
    
    # Create search domains for min and max
    min_domain = np.zeros((beat_qty, window), dtype=int)
    max_domain = np.zeros((beat_qty, window), dtype=int)
    
    for i in range(window):
        min_domain[:, i] = OT - i + 1
        max_domain[:, i] = OT + i - 1
    
    # Error protection
    min_domain[min_domain < 0] = 0
    max_domain[max_domain < 0] = 0
    max_domain[max_domain >= len(abp)] = len(abp) - 1
    
    # Find diastolic and systolic values
    abp_min_values = abp[min_domain]
    abp_max_values = abp[max_domain]
    
    P_dias = np.min(abp_min_values, axis=1)
    P_sys = np.max(abp_max_values, axis=1)
    
    # Get indices for timing
    dindex = np.argmin(abp_min_values, axis=1)
    sindex = np.argmax(abp_max_values, axis=1)
    
    # Convert to actual time indices
    dias_time = min_domain[np.arange(beat_qty), dindex]
    sys_time = max_domain[np.arange(beat_qty), sindex]
    
    # Pulse Pressure [mmHg]
    PP = P_sys - P_dias
    
    # Beat Period [samples]
    beat_period = np.diff(onset_times)
    
    # Mean pressure and negative derivative calculation
    dyneg = np.diff(abp)
    dyneg[dyneg > 0] = 0
    
    MAP = np.zeros(beat_qty)
    stddev = np.zeros(beat_qty)
    mean_dyneg = np.zeros(beat_qty)
    
    for i in range(beat_qty):
        # Mean arterial pressure for this beat
        interval = abp[onset_times[i]:onset_times[i+1]]
        MAP[i] = np.mean(interval)
        stddev[i] = np.std(interval)
        
        # Mean negative derivative
        dyneg_interval = dyneg[onset_times[i]:onset_times[i+1]]
        dyneg_interval = dyneg_interval[dyneg_interval != 0]  # Remove zeros
        if len(dyneg_interval) == 0:
            mean_dyneg[i] = 0
        else:
            mean_dyneg[i] = np.mean(dyneg_interval)
    
    # Systolic Area calculation using 0.3*sqrt(RR)
    RR = beat_period / fs  # RR time in seconds
    sys_duration = 0.3 * np.sqrt(RR)
    end_of_sys1 = np.round(OT + sys_duration * fs).astype(int)
    
    # Ensure end_of_sys1 doesn't exceed signal length
    end_of_sys1 = np.minimum(end_of_sys1, len(abp) - 1)
    
    sys_area1 = localfun_area(abp, OT, end_of_sys1, P_dias, fs)
    
    # Systolic Area calculation using 'first minimum slope' method
    slope_window = 35
    ST = end_of_sys1.copy()
    
    # Error protection
    ST[ST > (len(abp) - 35)] = len(abp) - 35
    
    # Create slope domain
    slope_domain = np.zeros((beat_qty, slope_window), dtype=int)
    for i in range(slope_window):
        slope_domain[:, i] = ST + i
    
    # Ensure indices are within bounds
    slope_domain[slope_domain >= len(abp)] = len(abp) - 1
    
    # Calculate slopes (derivative)
    abp_slope_values = abp[slope_domain]
    slope = np.diff(abp_slope_values, axis=1)
    slope[slope > 0] = 0  # Remove positive slopes
    
    # Find minimum absolute slope
    index = np.argmin(np.abs(slope), axis=1)
    end_of_sys2 = slope_domain[np.arange(beat_qty), index]
    sys_area2 = localfun_area(abp, OT, end_of_sys2, P_dias, fs)

    # ADDED calculate HR
    HR = 60*fs/beat_period
    
    # Create output array with all 12 features
    features = np.column_stack([
        sys_time,       # 0: Time of systole [samples]
        P_sys,          # 1: Systolic BP [mmHg]
        dias_time,      # 2: Time of diastole [samples]
        P_dias,         # 3: Diastolic BP [mmHg]
        PP,             # 4: Pulse pressure [mmHg]
        MAP,            # 5: Mean pressure [mmHg]
        beat_period,    # 6: Beat Period [samples]
        mean_dyneg,     # 7: mean_dyneg
        end_of_sys1,    # 8: End of systole time 0.3*sqrt(RR) method
        sys_area1,      # 9: Area under systole 0.3*sqrt(RR) method
        end_of_sys2,    # 10: End of systole time 1st min-slope method
        sys_area2,       # 11: Area under systole 1st min-slope method
        HR
    ])
    
    return features

def localfun_area(abp, onset, end_sys, P_dias, fs=125):
    """
    Helper function to calculate systolic area.
    From physionet: https://physionet.org/content/cardiac-output/1.0.0/code/2analyze/abpfeature.m
    
    Args:
        abp: ABP signal
        onset: Onset times
        end_sys: End of systole times
        P_dias: Diastolic pressures
        fs: Sampling frequency
    
    Returns:
        sys_area: Systolic area [mmHg*sec]
    """
    beat_qty = len(onset)
    sys_area = np.zeros(beat_qty)
    
    for i in range(beat_qty):
        # Ensure indices are within bounds
        start_idx = max(0, onset[i])
        end_idx = min(len(abp), end_sys[i] + 1)
        
        if end_idx > start_idx:
            sys_area[i] = np.sum(abp[start_idx:end_idx])
        else:
            sys_area[i] = 0
    
    sys_period = end_sys - onset
    
    # Time scale and subtract the diastolic area under each systolic interval
    sys_area = (sys_area - P_dias * sys_period) / fs  # Area [mmHg*sec]
    
    return sys_area

def jSQI_complete(features, onset, abp, fs=125):
    """
    ABP waveform signal quality index calculation. From physionet: https://physionet.org/content/cardiac-output/1.0.0/code/2analyze/jSQI.m
    
    Args:
        features: Features extracted from ABP (nx12 array from abpfeature function)
        onset: Onset times of ABP beats
        abp: Arterial blood pressure waveform
        fs: Sampling frequency (default 125 Hz)
    
    Returns:
        BeatQ: SQI of each beat (nx10 array): 0=good, 1=bad
            Col 0: logical OR of cols 1 thru 9
            Col 1: P not physiologic (<20 or >300 mmHg)
            Col 2: MAP not physiologic (<30 or >200 mmHg)
            Col 3: HR not physiologic (<20 or >200 bpm)
            Col 4: PP not physiologic (<20 mmHg)
            Col 5: abnormal Psys (beat-to-beat change > 20 mmHg)
            Col 6: abnormal Pdias (beat-to-beat change > 20 mmHg)
            Col 7: abnormal period (beat-to-beat change > 1/2 sec)
            Col 8: abnormal P(onset) (beat-to-beat change > 20 mmHg)
            Col 9: noisy beat (mean of negative dP < -3)
        r: fraction of good beats in ABP
    """
    
    if len(onset) < 30:
        return np.array([]), None
    
    # Thresholds
    range_P = [20, 300]      # mmHg
    range_MAP = [30, 200]    # mmHg
    range_HR = [20, 200]     # bpm
    range_PP = [20, np.inf]  # mmHg
    
    d_Psys = 20
    d_Pdias = 20
    d_Period = 62.5          # 62.5 samples = 1/2 second at 125Hz
    d_P_onset = 20
    
    noise_threshold = -3
    
    # Extract ABP features from the complete feature set
    Psys = features[:, 1]        # Systolic BP
    Pdias = features[:, 3]       # Diastolic BP
    PP = features[:, 4]          # Pulse pressure
    MAP = features[:, 5]         # Mean pressure
    beat_period = features[:, 6] # Beat Period
    mean_dyneg = features[:, 7]  # Mean negative derivative
    
    # Calculate heart rate
    HR = 60 * fs / beat_period
    
    # Absolute thresholding (flag unphysiologic beats)
    bad_P = np.where((Pdias < range_P[0]) | (Psys > range_P[1]))[0]
    bad_MAP = np.where((MAP < range_MAP[0]) | (MAP > range_MAP[1]))[0]
    bad_HR = np.where((HR < range_HR[0]) | (HR > range_HR[1]))[0]
    bad_PP = np.where(PP < range_PP[0])[0]
    
    # First difference thresholding (flag beat-to-beat variations)
    jerk_Psys = np.where(np.abs(np.diff(Psys)) > d_Psys)[0] + 1  # +1 because diff reduces length
    jerk_Pdias = np.where(np.abs(np.diff(Pdias)) > d_Pdias)[0]
    jerk_Period = np.where(np.abs(np.diff(beat_period)) > d_Period)[0] + 1
    
    # For onset pressure changes, we need pressure values at onset times
    onset_pressures = abp[onset[:-1]]  # Exclude last onset
    jerk_P_onset = np.where(np.abs(np.diff(onset_pressures)) > d_P_onset)[0]
    
    # Noise detector
    noisy = np.where(mean_dyneg < noise_threshold)[0]
    
    # Initialize SQI matrix
    n_beats = len(features)  # Number of beats with features
    bq = np.zeros((n_beats, 10), dtype=int)
    
    # Set flags for bad beats (handle index bounds)
    if len(bad_P) > 0:
        bad_P = bad_P[bad_P < n_beats]
        bq[bad_P, 1] = 1
        
    if len(bad_MAP) > 0:
        bad_MAP = bad_MAP[bad_MAP < n_beats]
        bq[bad_MAP, 2] = 1
        
    if len(bad_HR) > 0:
        bad_HR = bad_HR[bad_HR < n_beats]
        bq[bad_HR, 3] = 1
        
    if len(bad_PP) > 0:
        bad_PP = bad_PP[bad_PP < n_beats]
        bq[bad_PP, 4] = 1
        
    if len(jerk_Psys) > 0:
        jerk_Psys = jerk_Psys[jerk_Psys < n_beats]
        bq[jerk_Psys, 5] = 1
        
    if len(jerk_Pdias) > 0:
        jerk_Pdias = jerk_Pdias[jerk_Pdias < n_beats]
        bq[jerk_Pdias, 6] = 1
        
    if len(jerk_Period) > 0:
        jerk_Period = jerk_Period[jerk_Period < n_beats]
        bq[jerk_Period, 7] = 1
        
    if len(jerk_P_onset) > 0:
        jerk_P_onset = jerk_P_onset[jerk_P_onset < n_beats]
        bq[jerk_P_onset, 8] = 1
        
    if len(noisy) > 0:
        noisy = noisy[noisy < n_beats]
        bq[noisy, 9] = 1
    
    # Overall quality flag (OR of all individual flags)
    bq[:, 0] = np.logical_or.reduce(bq[:, 1:], axis=1).astype(int)
    
    # Make all "...101..." patterns into "...111..." (fill isolated good beats)
    y = bq[:, 0].copy()
    if len(y) >= 3:
        # Find where second difference equals 2 (101 pattern)
        diff2 = np.diff(y, n=2)
        isolated_good = np.where(diff2 == 2)[0] + 1  # +1 for indexing correction
        y[isolated_good] = 1
        bq[:, 0] = y
    
    BeatQ = bq.astype(bool)
    
    # Fraction of good beats overall
    r = np.sum(bq[:, 0] == 0) / len(onset)
    
    return BeatQ, r

def preprocess_abp_signal(abp_signal, fs):
    """
    ABP preprocessing with all features and quality assessment.

    returns averaged sbp, dbp, map, hr for given abp_signal

    """
        
    # Use onset detector for beat detection
    onset_indices = wabp_onset_detector(abp_signal, fs)
    
    if len(onset_indices) < 30:  # Need at least 30 beats for quality assessment
        return np.nan, np.nan, np.nan, np.nan
    
    # Extract complete features (all 12 features)
    features = abp_features(abp_signal, onset_indices, fs)
    
    if len(features) == 0:
        return np.nan, np.nan, np.nan, np.nan
    
    # Assess signal quality
    beat_quality, quality_ratio = jSQI_complete(features, onset_indices, abp_signal, fs)
    
    if beat_quality is None:
        return np.nan, np.nan, np.nan, np.nan
    
    # Filter out bad quality beats
    good_beats = ~beat_quality[:, 0]  # Use overall quality flag
    
    if np.sum(good_beats) < 5:  # Need at least 5 good beats
        return np.nan, np.nan, np.nan, np.nan
    # Extract systolic and diastolic times for good beats only
    systolic_bps = features[good_beats, 1]
    diastolic_bps = features[good_beats, 3]
    map_bps = (1/3) * systolic_bps + (2/3) * diastolic_bps

    median_sys = np.median(systolic_bps)
    median_dias = np.median(diastolic_bps)
    median_maps = np.median(map_bps)

    median_hr = np.mean(features[good_beats, 12])

    
    return median_sys, median_dias, median_maps, median_hr

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()